# ExRai-AMPKAR Z-stack visualization and heatmaps

This notebook generates Z-stack videos for ExRai-AMPKAR datasets acquired at five time points (0, 1, 3, 6, and 8 minutes). It combines the Ex800 and Ex920 image stacks with cell tracking masks and final ratiometric values obtained from the previous analysis pipeline.

For each time point, the script first generates a three-panel video showing the two excitation channels and a pseudo-colored ratio image. It then generates a second video with the corresponding background-subtracted intensity heatmaps below the image panels. Cells are matched between excitation channels using the final S7 correspondence table, and their maximum-intensity Z-plane is identified independently for each channel using the S5 background-subtracted values. The peak plane is highlighted on the heatmap and shown as a colored outline on the corresponding image channel.

The notebook uses previously calculated ratios and intensity measurements. It does not repeat segmentation, cell matching, background subtraction, or ratiometric analysis.


## Installation

Required packages can be installed with:

```bash
pip install numpy pandas matplotlib tifffile opencv-python pillow
```


In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import tifffile
import cv2
import matplotlib as mpl
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
from matplotlib import font_manager
import colorsys
from PIL import Image as PILImage


## Load datasets and visualization settings

Run the notebook from the directory containing the `0min`, `1min`, `3min`, `6min`, and `8min` folders. Each folder must contain the original `800_ch2_Z*.tif` and `920_ch2_Z*.tif` images and the `results` folder generated by the analysis pipeline.

The display intensity ranges for Ex800 and Ex920 are defined separately and kept fixed across time points. The ratio range and color palette are also fixed so that the same ratio always receives the same pseudo-color. Video frame rate, fonts, panel margins, and preview behavior can be adjusted here. These settings affect visualization only, not the calculated fluorescence values.


In [ ]:
# -------- Input folders and time points --------
# Load the analysis results from the Input_analysis_results folder.
BASE_DIR = Path.cwd() / "Input analysis results"

# Save all generated videos and CSV files in a separate output folder.
OUTPUT_DIR = Path.cwd() / "Output folder"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TIMEPOINTS = ["0min", "1min", "3min", "6min", "8min"]

# Image display only: (minimum, maximum) in source TIFF intensity units.
# Adjust these ranges to change channel brightness in the videos only.
EX800_RANGE = (0, 180)
EX920_RANGE = (5, 80)
# Video playback speed (frames per second).
FPS = 2.5

# Palette and default ratio range.
# Ratio values outside this range use the endpoint colors.
RATIO_RANGE = (0.2, 0.5)
COLOR_STOPS = ["#006400", "#66A05B", "#FFA500", "#FF8C00"]

# Display settings (video and inline player).
# -------- Video appearance --------
# Adjust labels, margins, and colorbar spacing here.
FONT_FAMILY = "Arial"     # Uses Arial if installed; falls back to a clean sans-serif
HEADER_HEIGHT = 110
LEFT_MARGIN = 24  # match the colorbar gap on the right for visual balance
BAR_WIDTH = 26
BAR_GAP = 24
BAR_LABEL_WIDTH = 105

# Set to False to skip the interactive preview while still saving AVI files.
SHOW_NOTEBOOK_PREVIEW = True

assert FPS > 0
assert EX800_RANGE[1] > EX800_RANGE[0] and EX920_RANGE[1] > EX920_RANGE[0]
assert RATIO_RANGE[1] > RATIO_RANGE[0]
# Use one ratio-to-color mapping for all frames and time points.
CMAP = LinearSegmentedColormap.from_list("ExRai_S7", COLOR_STOPS)
NORM = mpl.colors.Normalize(*RATIO_RANGE, clip=True)


## Load images, cell ratios, and tracking masks

For each time point, the function reads the final S7 ratio table and keeps cells with finite `ratio_920_over_800` values. Each Ex800 cell ID is linked to its stored ratio and corresponding Ex800 tracking mask from S2.

The TIFF images are ordered by their Z-plane number. Ex920 slices are sampled across the Ex800 frame sequence so both excitation channels can be displayed together even when their stacks contain different numbers of planes. The function also checks that image dimensions and mask stacks are compatible before rendering.


In [3]:
def load_timepoint(label):
    # Find the processed data for this time point.
    folder = BASE_DIR / label
    ratio_path = folder / "results/S7_final_output/S7_manual_ratio_results.csv"
    if not ratio_path.exists():
        raise FileNotFoundError(ratio_path)
    # Keep only valid S7 ratios and require one value per Ex800 cell.
    table = pd.read_csv(ratio_path)
    required = {"track_800", "ratio_920_over_800"}
    if not required.issubset(table):
        raise ValueError(f"Missing ratio columns: {required - set(table)}")
    table = table[np.isfinite(table["ratio_920_over_800"])].copy()
    if table["track_800"].duplicated().any():
        raise ValueError("Duplicate cell IDs in S7 ratios")
    values = dict(zip(table.track_800.astype(int), table.ratio_920_over_800.astype(float)))

    # Find single-plane TIFF files and order them by Z number.
    def stack(wavelength):
        pattern = re.compile(rf"{wavelength}_ch2_Z(\d+)\.tif$", re.I)
        matches = [(int(m.group(1)), p) for p in folder.glob(f"{wavelength}_ch2_Z*.tif")
                   if (m := pattern.fullmatch(p.name))]
        if not matches:
            raise FileNotFoundError(f"No Ex{wavelength} TIFF stack in {folder}")
        return sorted(matches)

    ex800, ex920 = stack("800"), stack("920")
    # Align the Ex920 planes to the number of Ex800 video frames.
    sampled_920 = np.rint(np.linspace(0, len(ex920)-1, len(ex800))).astype(int)
    shape = tifffile.imread(ex800[0][1]).shape
    if len(shape) != 2 or tifffile.imread(ex920[0][1]).shape != shape:
        raise ValueError(f"Image shape mismatch in {folder}")
    # Load the tracked Ex800 mask stacks used for the pseudocolor panel.
    masks = {}
    for track in values:
        filename = folder / f"results/S2_tracks_800/800_track{track:04d}_maskstack.tif"
        if not filename.exists():
            continue
        mask = tifffile.imread(filename) > 0
        if mask.ndim != 3 or mask.shape[1:] != shape or mask.shape[0] < max(z for z, _ in ex800):
            raise ValueError(f"Invalid mask stack: {filename}")
        masks[track] = mask
    if not masks:
        raise ValueError(f"No matching cell masks in {folder}")
    return ex800, ex920, sampled_920, shape, values, masks


## Generate the three-panel Z-stack videos

The first set of videos displays three synchronized panels for every Ex800 Z-plane: the Ex800 image in green, the selected Ex920 image in yellow, and a pseudo-color image of the final Ex920/Ex800 ratio. The raw images are scaled using the display limits specified above.

In the ratio panel, each tracked cell retains its S7 ratio color across Z, while the mask follows the cell's segmentation in the current Ex800 plane. Gaussian smoothing is applied only to the displayed mask boundary. The same ratio colorbar is used for all time points.


In [4]:
def tint(image, color, bounds):
    # Scale the chosen intensity window to 0–1, then apply the channel color.
    lo, hi = bounds
    intensity = np.clip((image.astype(np.float32) - lo) / (hi - lo), 0, 1)
    return np.rint(intensity[..., None] * np.array(color)).astype(np.uint8)


### Text and panel formatting

 The renderer creates a fixed header, draws the three image panels with white borders, and places the ratio colorbar beside the images. All output frames have the same layout within each time point.


In [5]:
def _font(size):
    # Use the selected font when available, otherwise use a fallback.
    try:
        path = font_manager.findfont(FONT_FAMILY, fallback_to_default=False)
    except ValueError:
        path = font_manager.findfont("DejaVu Sans")
    return ImageFont.truetype(path, size=max(9, int(size)))


In [6]:
def make_renderer(label, data):
    ex800, ex920, sampled, (height, width), ratios, masks = data
    # Reserve space for three image panels, the header, and ratio colorbar.
    right_space = BAR_GAP + BAR_WIDTH + BAR_LABEL_WIDTH
    frame_width = LEFT_MARGIN + 3 * width + right_space
    frame_height = height + HEADER_HEIGHT
    bar_y0, bar_y1 = HEADER_HEIGHT + 15, HEADER_HEIGHT + height - 15
    bar_height = max(1, bar_y1 - bar_y0)
    # Assign each tracked cell a fixed color based on its final S7 ratio.
    colors = {track: (np.array(CMAP(NORM(ratio))[:3]) * 255).astype(np.uint8)
              for track, ratio in ratios.items() if track in masks}

    font_title = _font(min(40, max(26, width // 10)))
    font_channel = _font(min(34, max(23, width // 14)))
    font_tick = _font(21)

    # Fit and center multiline captions without clipping.
    def centered_text(draw, caption, x_center, y_top, max_width, font, fill="white"):
        """Center each line horizontally; center the multiline block vertically."""
        lines = caption.split("\n")
        # Fit the longest line without unnecessarily shrinking short labels.
        while max(draw.textbbox((0, 0), line, font=font)[2] -
                  draw.textbbox((0, 0), line, font=font)[0]
                  for line in lines) > max_width and font.size > 14:
            font = _font(font.size - 1)
        spacing = max(4, round(font.size * 0.15))
        heights = [draw.textbbox((0, 0), line, font=font)[3] -
                   draw.textbbox((0, 0), line, font=font)[1] for line in lines]
        total_height = sum(heights) + spacing * (len(lines) - 1)
        yy = y_top - total_height / 2
        for line, h in zip(lines, heights):
            box = draw.textbbox((0, 0), line, font=font)
            w = box[2] - box[0]
            draw.text((round(x_center - w / 2 - box[0]), round(yy - box[1])),
                      line, font=font, fill=fill)
            yy += h + spacing

    # Build one RGB frame from the corresponding Ex800 and Ex920 planes.
    def frame(i):
        z800, path800 = ex800[i]
        z920, path920 = ex920[sampled[i]]
        raw800, raw920 = tifffile.imread(path800), tifffile.imread(path920)
        if raw800.shape != (height, width) or raw920.shape != (height, width):
            raise ValueError(f"Unexpected image dimensions at {label}, frame {i}")
        # Fill cell masks with their ratio colors for the third panel.
        pseudo = np.zeros((height, width, 3), dtype=np.uint8)
        for track, mask in masks.items():
            # Smooth only the DISPLAY contour; analysis masks and ratio values are unchanged.
            plane = mask[z800 - 1].astype(np.uint8) * 255
            # Smooth the displayed mask boundary without changing the stored mask.
            plane = (cv2.GaussianBlur(plane, (0, 0), 2.0) >= 127).astype(np.uint8) * 255
            pseudo[plane > 0] = colors[track]

        # Place the two tinted channels beside the cell-ratio pseudocolor.
        out = np.zeros((frame_height, frame_width, 3), dtype=np.uint8)
        out[HEADER_HEIGHT:, LEFT_MARGIN:LEFT_MARGIN+width] = tint(raw800, (0, 255, 80), EX800_RANGE)
        out[HEADER_HEIGHT:, LEFT_MARGIN+width:LEFT_MARGIN+2*width] = tint(raw920, (255, 225, 30), EX920_RANGE)
        out[HEADER_HEIGHT:, LEFT_MARGIN+2*width:LEFT_MARGIN+3*width] = pseudo

        # Crisp white framing distinguishes the three image panels.
        for j in range(3):
            x_left = LEFT_MARGIN + j * width
            cv2.rectangle(out, (x_left, HEADER_HEIGHT),
                          (x_left + width - 1, frame_height - 1),
                          (255, 255, 255), 2)

        # Fixed colorbar for consistent pseudocolor comparisons among time points.
        x0 = LEFT_MARGIN + 3*width + BAR_GAP
        bar_colors = (CMAP(np.linspace(1, 0, bar_height))[:, :3] * 255).astype(np.uint8)
        out[bar_y0:bar_y0+bar_height, x0:x0+BAR_WIDTH] = bar_colors[:, None, :]
        cv2.rectangle(out, (x0, bar_y0), (x0+BAR_WIDTH, bar_y0+bar_height), (235,235,235), 1)
        for proportion, value in ((0, RATIO_RANGE[1]), (0.5, np.mean(RATIO_RANGE)), (1, RATIO_RANGE[0])):
            yy = min(frame_height-7, bar_y0 + round(proportion*bar_height))
            cv2.line(out, (x0+BAR_WIDTH, yy), (x0+BAR_WIDTH+6, yy), (255,255,255), 1)

        # Add titles, plane numbers, and colorbar labels.
        # PIL draws the labels on top of the assembled RGB frame.
        pil = Image.fromarray(out)
        draw = ImageDraw.Draw(pil)
        centered_text(draw, label.replace("min", " min"), LEFT_MARGIN + 1.5*width, 23,
                      3*width-16, font_title)
        panel_titles = (f"Ex 800  |  Plane {z800}",
                        f"Ex 920  |  Plane {z920}",
                        "Ratio(Ex 920/800)\npseudocolor")
        for j, caption in enumerate(panel_titles):
            centered_text(draw, caption, LEFT_MARGIN + (j+0.5)*width, 77,
                          width-12, font_channel)
        for proportion, value in ((0, RATIO_RANGE[1]), (0.5, np.mean(RATIO_RANGE)), (1, RATIO_RANGE[0])):
            yy = min(frame_height-7, bar_y0 + round(proportion*bar_height))
            draw.text((x0+BAR_WIDTH+11, max(HEADER_HEIGHT+5, min(frame_height-21, yy-10))),
                      f"{value:.2f}", font=font_tick, fill="white")
        return np.asarray(pil)

    return frame, (frame_width, frame_height), len(ex800)


### Save and preview the ratio videos

The next cell saves one AVI for each time point as `ExRai_<timepoint>_Zstack.avi`. Frames are generated sequentially and converted to the color order expected by OpenCV before being written to disk.

If notebook previews are enabled, the same frames are displayed as an interactive animation. The player retains controls for play/pause, frame selection, speed, and looping.


In [7]:
# -------- Export the three-panel videos --------
# Each time point is loaded and rendered independently.
for timepoint in TIMEPOINTS:
    data = load_timepoint(timepoint)
    make_frame, size, n_frames = make_renderer(timepoint, data)
    filename = OUTPUT_DIR / f"ExRai_{timepoint}_Zstack.avi"
    # AVI export uses the same frame size and FPS for all slices.
    writer = cv2.VideoWriter(str(filename), cv2.VideoWriter_fourcc(*"MJPG"), float(FPS), size)
    if not writer.isOpened():
        raise RuntimeError(f"AVI writer failed: {filename}")
    try:
        for i in range(n_frames):
            writer.write(cv2.cvtColor(make_frame(i), cv2.COLOR_RGB2BGR))
    finally:
        writer.release()
    print(f"{timepoint.replace('min', ' min')}: saved {filename.name}")

    # Optional interactive player; this does not affect the saved video.
    if SHOW_NOTEBOOK_PREVIEW:
        fig, ax = plt.subplots(figsize=(14, 5))
        artist = ax.imshow(make_frame(0), interpolation="nearest")
        ax.axis("off")
        fig.tight_layout(pad=0.1)
        def update(i):
            artist.set_data(make_frame(i))
            return (artist,)
        movie = FuncAnimation(fig, update, frames=n_frames, interval=1000/FPS,
                              blit=True, repeat=True)
        plt.close(fig)
        html = movie.to_jshtml(default_mode="loop")
        html, count = re.subn(
            r"(\b(anim[a-fA-F0-9]+)\s*=\s*new Animation\([\s\S]*?loop_select_id\);)",
            r"\1\n        \2.play_animation();",
            html, count=1,
        )
        if count != 1:
            raise RuntimeError("Matplotlib JS player initialization changed; cannot enable autoplay safely")
        display(HTML(html))
    del data, make_frame


0 min: saved ExRai_0min_Zstack.avi


1 min: saved ExRai_1min_Zstack.avi


3 min: saved ExRai_3min_Zstack.avi


6 min: saved ExRai_6min_Zstack.avi


8 min: saved ExRai_8min_Zstack.avi


## Generate background-subtracted heatmaps and peak-plane outlines

This second visualization uses the S5 per-cell background-subtracted intensity tables for both excitation channels. The Ex800 and Ex920 heatmaps are aligned by their final S7 cell correspondence: each column represents the same matched cell in both heatmaps, even though its track ID may differ between channels.

For each cell, the Z-plane with the highest finite background-subtracted intensity is determined separately for Ex800 and Ex920. The peak is outlined in the heatmap and, when displayed, on the corresponding excitation image. Ex800 and Ex920 each use their own fixed intensity scale. The resulting heatmaps are placed below the original three-panel video.


In [8]:
# -------- Peak and heatmap display settings --------
# Adjust these values to change spacing, outlines, and output names.
HEAT_CELL_WIDTH = 11             # pixels per tracked cell column
HEATMAP_GAP = 28                 # space after original ratio colorbar
HEATMAP_BETWEEN = 18             # gap between the two heatmaps
HEATMAP_RIGHT_MARGIN = 24
HEATMAP_TOP_PADDING = 5
PEAK_OUTLINE_THICKNESS = 2
OUTPUT_SUFFIX = "_video"


### Read S5 background-subtracted intensities

The S5 output files contain the mean background-subtracted intensity for each tracked cell at each Z-plane. The helper locates the correct `800_S5_means_bgsub.csv` or `920_S5_means_bgsub.csv` file by filename rather than assuming that the parent folder name always matches the channel.


In [9]:
def _s5_file(folder, channel):
    # Search by FILENAME, not directory name. Some exports have swapped
    # S5_backg_subtraction_800 / _920 folder names.
    # Locate the CSV by its channel-specific filename.
    paths = sorted((folder / "results").glob(
        f"S5_backg_subtraction_*/{channel}_S5_means_bgsub.csv"))
    if len(paths) != 1:
        raise FileNotFoundError(f"Expected exactly one Ex{channel} S5 CSV in {folder}; found {paths}")
    return paths[0]


### Match cells and identify maximum-intensity planes

The final S7 mapping establishes which Ex800 and Ex920 tracks belong to the same cell. The S5 tables are reorganized according to this mapping, ensuring that a column in the Ex800 heatmap corresponds to the same cell as that column in the Ex920 heatmap.

For every matched cell, the script identifies the Z-plane with the highest finite S5 intensity separately in each channel. Missing values are ignored for peak selection, and background-subtracted negative values are retained. If several planes share the same maximum, the first plane in the sorted stack is selected.

The corresponding Ex920 segmentation masks are loaded for outlines. Each cell receives a distinct display color used consistently for its outlines and peak markers within a time point. 


In [10]:
def _prepare_peaks(label, data):
    ex800, ex920, sampled, (h, w), ratios, masks800 = data
    folder = BASE_DIR / label
    # Use the manually validated S7 map to pair Ex800 and Ex920 cells.
    final = pd.read_csv(folder / "results/S7_final_output/S7_manual_ratio_results.csv")
    final = final.loc[final.track_800.isin(masks800)].copy()
    final = final.loc[final.ratio_920_over_800.notna()]
    if final.track_800.duplicated().any() or final.track_920.duplicated().any():
        raise ValueError(f"Non-unique 800->920 mapping at {label}; check manually validated S7 table")
    final = final.sort_values("track_800").reset_index(drop=True)
    if final.empty:
        raise ValueError(f"No eligible mapped cells for {label}")

    # Read the S5 background-subtracted intensity for every Z plane.
    heat = {}
    for channel in ("800", "920"):
        df = pd.read_csv(_s5_file(folder, channel)).sort_values("z_number")
        z = df.z_number.to_numpy(dtype=int)
        if len(np.unique(z)) != len(z):
            raise ValueError(f"Duplicate Z values in S5 {channel} at {label}")
        heat[channel] = (df, z)

    # Keep the same cell order in both channel heatmaps.
    ids800 = final.track_800.astype(int).tolist()
    ids920 = final.track_920.astype(int).tolist()
    matrices = {}
    peaks = {}
    for ch, ids in (("800", ids800), ("920", ids920)):
        df, zvals = heat[ch]
        missing = [f"cell_{tid}" for tid in ids if f"cell_{tid}" not in df]
        if missing:
            raise ValueError(f"Missing S5 {ch} columns for matched cells: {missing}")
        # SAME column index = SAME cell, using Ex800 ID as reference.
        mat = df[[f"cell_{tid}" for tid in ids]].to_numpy(dtype=float)
        matrices[ch] = mat
        # For each cell, find the plane with the highest valid intensity.
        zpeak = {}
        for column, t800 in enumerate(ids800):
            valid = np.flatnonzero(np.isfinite(mat[:, column]))
            if valid.size:
                # Earliest plane in the event of a tie. Preserve negative
                # background-subtracted values; do not replace NaN by zero.
                zpeak[t800] = int(zvals[valid[np.argmax(mat[valid, column])]])
        peaks[ch] = zpeak

    # Match the Ex920 masks back to the Ex800 reference cell IDs.
    masks920 = {}
    for t800, t920 in zip(ids800, ids920):
        path = folder / f"results/S2_tracks_920/920_track{t920:04d}_maskstack.tif"
        if path.exists():
            a = tifffile.imread(path) > 0
            if a.ndim == 3 and a.shape[1:] == (h, w):
                masks920[t800] = a
    # High-contrast, NON-REPEATING cell colors. Colors are chosen far from
    # the green/yellow fluorescence and viridis heatmap colors as possible.
    # Deterministic distinct palette, consistent for a track within a time point.
    # Edit this palette to change the peak-outline colors.
    candidate_hex = [
        "#FF00FF", "#00DFFF", "#FF3355", "#FFFFFF", "#8B5CFF",
        "#FF69B4", "#00AAFF", "#FF7A00", "#E7DBFF", "#E6194B",
        "#A33AFF", "#FFB3E6", "#37FFFF", "#FF4040", "#AB83FF",
        "#FAFA72", "#FF00B7", "#4D85FF", "#D500F9", "#FFA4A4",
    ]
    # For larger datasets use a golden-angle sequence rather than repeating.
    colors = {}
    for i, track in enumerate(ids800):
        if i < len(candidate_hex):
            hx = candidate_hex[i].lstrip("#")
            colors[track] = tuple(int(hx[k:k+2], 16) for k in (0,2,4))
        else:
            hue = (0.83 + (i-len(candidate_hex))*0.61803398875) % 1
            colors[track] = tuple(round(255*v) for v in colorsys.hsv_to_rgb(hue, .95, 1))

    # Video frames can repeat Ex920 planes if stacks have different lengths.
    # Designate ONE frame per Z plane, so every Ex920 cell outline appears
    # exactly once, even if sampled[] repeats a plane.
    # Map each Ex920 peak to a single video frame.
    chosen_frame920 = {}
    for t800, peak in peaks["920"].items():
        candidate = [i for i, j in enumerate(sampled) if ex920[int(j)][0] == peak]
        if candidate:
            chosen_frame920[t800] = candidate[len(candidate)//2]
        else:
            # If not sampled at all, closest available Ex920 plane is used
            # for DISPLAY only; the heatmap retains the true peak Z.
            chosen_frame920[t800] = int(np.argmin([
                abs(ex920[int(j)][0] - peak) for j in sampled]))
            print(f"{label}: Ex920 track {t800} peak Z{peak} not sampled; "
                  f"outline shown at nearest displayed Ex920 plane")

    return ids800, colors, matrices, heat, peaks, masks920, chosen_frame920


### Convert intensity tables to heatmap images

The background-subtracted intensity matrix contains Z-planes as rows and matched cells as columns. The Ex800 values are displayed on a fixed scale of 0–100 and Ex920 values on a fixed scale of 0–30. Intensity clipping affects the displayed grayscale only; the original S5 data remain unchanged. Non-finite entries receive a separate dark color.

The matrix is converted into a raster image by assigning a display rectangle to each cell and Z-plane. This allows the heatmaps to be resized and aligned with the rest of the video.


In [11]:
def _heatmap_raster(mat, target_height, channel, cell_px=HEAT_CELL_WIDTH):
    # Fixed background-subtracted intensity range across all time points
    # Change these bounds to adjust heatmap contrast, not peak detection.
    if channel == "800":
        vmin, vmax = 0.0, 100.0
    elif channel == "920":
        vmin, vmax = 0.0, 30.0
    else:
        raise ValueError(f"Unknown channel: {channel}")

    # Clip only for visualization; preserve original data
    # Normalize each channel using its fixed display range.
    norm = np.clip((mat - vmin) / (vmax - vmin), 0, 1)
    rgb = (
        mpl.colormaps["Greys_r"](
            np.nan_to_num(norm, nan=0)
        )[:, :, :3] * 255
    ).astype(np.uint8)
    # Show missing measurements in dark gray.
    rgb[~np.isfinite(mat)] = (28, 28, 28)
    # Give each Z plane a row and each cell a column in the raster.
    z_edges = np.rint(
        np.linspace(0, target_height, mat.shape[0] + 1)
    ).astype(int)
    raster = np.zeros(
        (target_height, mat.shape[1] * cell_px, 3),
        dtype=np.uint8
    )
    for zi in range(mat.shape[0]):
        for ci in range(mat.shape[1]):
            raster[
                z_edges[zi]:z_edges[zi + 1],
                ci * cell_px:(ci + 1) * cell_px
            ] = rgb[zi, ci]
    return raster, z_edges, (vmin, vmax)


### Draw cell outlines at their peak planes

A colored contour is drawn on the corresponding Ex800 or Ex920 image only when that cell's selected peak plane is displayed. Gaussian smoothing is applied to the displayed outline mask before contour extraction; the stored segmentation masks are not modified.


In [12]:
def _overlay_cell_outline(frame, mask, xoff, yoff, rgb):
    plane = (mask.astype(np.uint8) * 255)
    # Smooth the display mask before extracting its contour.
    plane = (cv2.GaussianBlur(plane, (0, 0), 2.0) >= 127).astype(np.uint8)*255
    if not np.any(plane):
        return
    # Draw the contour over the original channel panel at its offset.
    contours, _ = cv2.findContours(plane, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    region = frame[yoff:yoff+plane.shape[0], xoff:xoff+plane.shape[1]]
    cv2.drawContours(region, contours, -1, rgb, PEAK_OUTLINE_THICKNESS, lineType=cv2.LINE_AA)

### Heatmap layout and labels

The heatmaps share the available width below the three original panels. Layout parameters control their height, spacing, row labels, and channel-specific intensity colorbars. Row labels identify the Z-plane positions in the S5 tables; columns follow the matched-cell ordering defined above.


In [13]:
# -------- Bottom heatmap layout --------
# These settings control the heatmap size and spacing below the panels.
HEATMAP_HEIGHT = 140
HEATMAP_TOP_GAP = 80
HEATMAP_BOTTOM_GAP = 20
HEATMAP_SIDE_GAP = 20
HEATMAP_ROW_LABEL_W = 30
HEATMAP_COLORBAR_H = 15
HEATMAP_COLORBAR_GAP = 20
HEATMAP_BOTTOM_MARGIN = 30


### Position heatmap titles

This helper centers the heatmap titles, intensity labels, and Z-row numbers using the same font-rendering approach as the image panels.


In [14]:
def _draw_centered(draw, caption, cx, cy, font, color="white"):
    caption = str(caption)

    # Measure the full label before placing it at the requested center.
    bounds = draw.multiline_textbbox(
        (0, 0), caption, font=font, spacing=4, align="center"
    )

    x = cx - (bounds[0] + bounds[2]) / 2
    y = cy - (bounds[1] + bounds[3]) / 2

    draw.multiline_text(
        (round(x), round(y)),
        caption,
        fill=color,
        font=font,
        spacing=4,
        align="center"
    )


### Combine image panels, peak outlines, and heatmaps

For each video frame, the renderer first draws the original Ex800, Ex920, and ratio panels. It then adds the peak-plane outlines to the individual excitation images using the matching cell colors. Underneath, it places the two background-subtracted intensity heatmaps, with colored rectangles highlighting each cell's maximum-intensity Z-plane.

The heatmap columns stay aligned between channels, while the Ex800 and Ex920 colorbars retain their own intensity limits. A separate CSV, `ExRai_<timepoint>_peak_alignment_v5.csv`, records the matched track IDs, peak Z-planes, and display colors for later inspection.


In [15]:
def make_peak_heatmap_renderer(label, data):
    # Reuse the existing three-panel renderer and add peak heatmaps below.
    base_frame, (base_w, base_h), nframes = make_renderer(label, data)
    ex800, ex920, sampled, (h, w), ratios, masks800 = data
    ids, colors, mats, heats, peaks, masks920, frame920 = _prepare_peaks(label, data)
    # Make heatmap columns wide enough to fill the entire width underneath
    # the existing THREE panels AND the original ExRai ratio colorbar.
    entire_w = base_w
    margin = LEFT_MARGIN
    HEATMAP_WIDTH_FACTOR = 0.75  # Adjust here (0.5–1.0)
    available_w = entire_w - 2*margin - HEATMAP_SIDE_GAP
    group_w = int(available_w * HEATMAP_WIDTH_FACTOR / 2)
    heat_w = group_w - HEATMAP_ROW_LABEL_W

    # Assign each matched cell an aligned column in both heatmaps.
    col_bounds = np.rint(np.linspace(0, heat_w, len(ids)+1)).astype(int)
    heat_y = base_h + HEATMAP_TOP_GAP
    bar_y = heat_y + HEATMAP_HEIGHT + HEATMAP_COLORBAR_GAP
    out_h = bar_y + HEATMAP_COLORBAR_H + HEATMAP_BOTTOM_MARGIN
    # Center both heatmaps as a group
    total_group_width = 2 * group_w + HEATMAP_SIDE_GAP  
    # Adjust this offset to move both heatmaps left or right.
    HEATMAP_SHIFT_X = -70  # Negative = left; positive = right
    start_x = (entire_w - total_group_width) // 2 + HEATMAP_SHIFT_X    

    centers = {
        "800": start_x + group_w/2,
        "920": start_x + group_w + HEATMAP_SIDE_GAP + group_w/2
    }
    
    x_heat = {
        "800": start_x + HEATMAP_ROW_LABEL_W,
        "920": start_x + group_w + HEATMAP_SIDE_GAP + HEATMAP_ROW_LABEL_W
    }
    fs = _font(19)
    tickfont = _font(14)
    titlefont = _font(20)

    # Precompute the heatmap images so only overlays change per frame.
    heat_rasters, row_edges, scales = {}, {}, {}
    for ch in ("800", "920"):
        raster, ed, scale = _heatmap_raster(mats[ch],HEATMAP_HEIGHT,channel=ch,cell_px=1)
        # Rebuild each cell column with precisely aligned integer edges
        x = np.zeros((HEATMAP_HEIGHT, heat_w, 3), np.uint8)
        for j in range(len(ids)):
            x[:, col_bounds[j]:col_bounds[j+1]] = raster[:, j:j+1]
        heat_rasters[ch], row_edges[ch], scales[ch] = x, ed, scale

    # Combine one video frame with peak outlines and the two heatmaps.
    def render(i):
        out = np.zeros((out_h, entire_w, 3), dtype=np.uint8)
        out[:base_h, :] = base_frame(i)
        z800 = int(ex800[i][0]); z920 = int(ex920[int(sampled[i])][0])
        # Only show each peak cell outline in its OWN excitation channel,
        # and only once in the video at that channel's peak plane.
        for t800 in ids:
            rgb = colors[t800]
            if peaks["800"].get(t800) == z800:
                m = masks800[t800]
                if 0 <= z800-1 < len(m):
                    _overlay_cell_outline(out, m[z800-1], LEFT_MARGIN, HEADER_HEIGHT, rgb)
            if frame920.get(t800) == i and t800 in masks920:
                m = masks920[t800]
                if 0 <= z920-1 < len(m):
                    _overlay_cell_outline(out, m[z920-1], LEFT_MARGIN+w, HEADER_HEIGHT, rgb)

        # Label the bottom panels before inserting their heatmap rasters.
        pil_pre = PILImage.fromarray(out)
        dd = ImageDraw.Draw(pil_pre)
        for ch in ("800", "920"):
            cx = x_heat[ch] + heat_w / 2
        
            _draw_centered(
                dd, f"Ex {ch} intensity", cx,
                base_h + HEATMAP_TOP_GAP/2 - 12,
                titlefont
            )
        
            _draw_centered(
                dd, "(max plane highlighted)",
                cx,
                base_h + HEATMAP_TOP_GAP/2 + 14,
                fs
            )        
        
        out = np.array(pil_pre)

        # Highlight each cell’s maximum-intensity row with its outline color.
        for ch in ("800", "920"):
            x0 = x_heat[ch]
            mat = mats[ch]
            ed = row_edges[ch]
            out[heat_y:heat_y+HEATMAP_HEIGHT, x0:x0+heat_w] = heat_rasters[ch]
            # Frame peak-intensity heatmap block in the SAME cell-specific color.
            for ci, tid in enumerate(ids):
                zz = peaks[ch].get(tid)
                if zz is None: continue
                zr = np.flatnonzero(heats[ch][1] == zz)
                if not len(zr): continue
                j = int(zr[0])
                ya, yb = heat_y + int(ed[j]), heat_y + int(ed[j+1])-1
                xa, xb = x0+int(col_bounds[ci]), x0+int(col_bounds[ci+1])-1
                if xb >= xa and yb >= ya:
                    cv2.rectangle(out, (xa,ya),(xb,yb),colors[tid], 2, lineType=cv2.LINE_AA)
            cv2.rectangle(out, (x0,heat_y),
                          (x0+heat_w-1,heat_y+HEATMAP_HEIGHT-1), (255,255,255),1)
            # Independent intensity scale for this channel.
            grad = (mpl.colormaps["Greys_r"](np.linspace(0,1,heat_w))[:,:3]*255).astype(np.uint8)
            out[bar_y:bar_y+HEATMAP_COLORBAR_H, x0:x0+heat_w] = grad[None,:,:]
            cv2.rectangle(out, (x0,bar_y), (x0+heat_w-1,bar_y+HEATMAP_COLORBAR_H-1),
                          (240,240,240),1)

        # Add Z-row labels and the separate channel-intensity scales.
        pil = PILImage.fromarray(out)
        draw = ImageDraw.Draw(pil)
        for ch in ("800", "920"):
            x0=x_heat[ch]
            ed=row_edges[ch]
            for zi in range(mats[ch].shape[0]):
                cy = heat_y + (int(ed[zi])+int(ed[zi+1]))/2
                # Requested row labels: 1,2,3,4... (ascending top to bottom).
                _draw_centered(draw, zi+1, x0-HEATMAP_ROW_LABEL_W/2,cy,tickfont)
            mn,mx=scales[ch]
            _draw_centered(draw, f"{mn:.1f}",x0,bar_y+HEATMAP_COLORBAR_H+15,tickfont)
            _draw_centered(draw, f"{(mn+mx)/2:.1f}", x0+heat_w/2,
                           bar_y+HEATMAP_COLORBAR_H+15,tickfont)
            _draw_centered(draw, f"{mx:.1f}",x0+heat_w,
                           bar_y+HEATMAP_COLORBAR_H+15,tickfont)
        return np.asarray(pil)

    
    # Save the cell correspondences, peak planes, and displayed outline colors.
    mapping = pd.DataFrame({
        "track_800": ids,
        "track_920": [int(x) for x in pd.read_csv(
            BASE_DIR/label/"results/S7_final_output/S7_manual_ratio_results.csv").set_index(
                "track_800").loc[ids,"track_920"]],
        "peak_Z800": [peaks["800"].get(t,np.nan) for t in ids],
        "peak_Z920": [peaks["920"].get(t,np.nan) for t in ids],
        "RGB_color": [str(colors[t]) for t in ids],
    })
    mapping.to_csv(OUTPUT_DIR / f"ExRai_{label}_peak_alignment.csv", index=False)
    return render, (entire_w,out_h), nframes


### Save and preview the heatmap videos

The final cell writes one AVI per time point with the suffix `_peaks_heatmaps_bottom_v5.avi`, without replacing the original three-panel videos. A PNG preview of the first frame at 0 minutes is also saved. When notebook previews are enabled, each heatmap video is displayed using the same interactive animation controls as above.


In [16]:
# -------- Export the peak-heatmap videos --------
# Rendering also saves a CSV with the matched cell and peak-plane information.
for timepoint in TIMEPOINTS:
    data = load_timepoint(timepoint)
    renderer, size, nframes = make_peak_heatmap_renderer(timepoint,data)
    path = OUTPUT_DIR / f"ExRai_{timepoint}_Zstack{OUTPUT_SUFFIX}.avi"
    # Write each RGB frame to an AVI file in OpenCV’s BGR format.
    writer = cv2.VideoWriter(str(path),cv2.VideoWriter_fourcc(*"MJPG"),float(FPS),size)
    if not writer.isOpened(): raise RuntimeError(f"Cannot write {path}")
    try:
        for fi in range(nframes):
            writer.write(cv2.cvtColor(renderer(fi),cv2.COLOR_RGB2BGR))
    finally:
        writer.release()
    print(f"Saved {path.name}; dimensions {size}")
    # Show the interactive animation only when previews are enabled.
    if SHOW_NOTEBOOK_PREVIEW:
        fig, ax = plt.subplots(figsize=(17, 10))
        artist = ax.imshow(renderer(0),interpolation="nearest")
        ax.axis("off")
        fig.tight_layout(pad=.1)
        def update(fi):
            artist.set_data(renderer(fi));return (artist,)
        movie=FuncAnimation(fig,update,frames=nframes,interval=1000/FPS,blit=True,repeat=True)
        plt.close(fig)
        html=movie.to_jshtml(default_mode="loop")
        html,nn = re.subn(r"(\b(anim[a-fA-F0-9]+)\s*=\s*new Animation\([\s\S]*?loop_select_id\);)",
                          r"\1\n        \2.play_animation();",html,count=1)
        if nn != 1: raise RuntimeError("Autoplay initialization not found")
        display(HTML(html))
    del renderer, data


Saved ExRai_0min_Zstack_video.avi; dimensions (719, 575)


Saved ExRai_1min_Zstack_video.avi; dimensions (719, 575)


Saved ExRai_3min_Zstack_video.avi; dimensions (719, 575)


Saved ExRai_6min_Zstack_video.avi; dimensions (719, 575)


Saved ExRai_8min_Zstack_video.avi; dimensions (719, 575)
